# NBA / NHL team model training (Colab GPU tier)

Heavy compute (the MLP / deep-learning family) runs here, not on the laptop. Flow:
1. Export game features locally, once per sport:
   `python run.py --export-features --sport nba` and `--sport nhl`
2. Upload `data/exports/nba_game_features.parquet` and `nhl_game_features.parquet` to your Drive folder `sports-edge/exports/`
3. Run this notebook on a **GPU runtime** (Runtime -> Change runtime type -> GPU)
4. Trained artifacts are written back to Drive `sports-edge/artifacts/`
5. Download them locally and run `python run.py --import-models`

The exported parquet already contains the injury / WOWY features, so no DB or API keys are needed here.


In [ ]:
# 1. Install deps (LightGBM is preinstalled on Colab; add the heavy extras)
!pip -q install torch xgboost lightgbm polars scikit-learn pyarrow


In [ ]:
# 2. Mount Drive and locate the exported features
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/sports-edge'
EXPORTS = os.path.join(BASE, 'exports')
ARTIFACTS = os.path.join(BASE, 'artifacts')
os.makedirs(ARTIFACTS, exist_ok=True)
FEATURES = {s: os.path.join(EXPORTS, f'{s}_game_features.parquet') for s in ('nba', 'nhl')}
for s, p in FEATURES.items():
    print(s, 'present:', os.path.exists(p))


In [ ]:
# 3. Get the project code so models/compare_team.py and models/train.py import.
#    Clone your repo (or upload the package) and put it on the path:
# !git clone <your-repo-url> /content/sports-edge
# %cd /content/sports-edge
import sys; sys.path.insert(0, '/content/sports-edge')


In [ ]:
# 4. Confirm GPU is visible to PyTorch
import torch
print('CUDA available:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


## Moneyline bake-off (LightGBM / XGBoost / MLP / ensemble)

Each sport gets its own comparison. The MLP (deep-learning model) is trained and
scored on the GPU inside `run_team_comparison`; the winning **portable** model is
saved to `{sport}_moneyline.pkl`.


In [ ]:
from models.compare_team import run_team_comparison
tables = {}
for sport in ('nba', 'nhl'):
    p = FEATURES[sport]
    if not os.path.exists(p):
        print(f'skip {sport}: {p} not uploaded'); continue
    print(f'\n=== {sport.upper()} bake-off ===')
    tables[sport] = run_team_comparison(sport, features_path=p)
for s, t in tables.items():
    print(s); display(t)


## Copy artifacts back to Drive

Then download `{sport}_moneyline.pkl` and `{sport}_model_comparison.csv` into
`data/exports/artifacts/` locally and run `python run.py --import-models`.


In [ ]:
import shutil, glob
for pat in ('nba_*', 'nhl_*'):
    for f in glob.glob(f'models/artifacts/{pat}'):
        shutil.copy(f, ARTIFACTS)
print('Copied artifacts to', ARTIFACTS)
